# Cache-first lead-time teleconnection metrics

This notebook diagnoses lead-dependent teleconnections between one upstream SST index and one downstream gridded variable. It shares preparation contracts and reusable processors with:

- `3a_refactor_sst_skill_ts.ipynb`: compact E3SM/HadISST2 regional SST-index time series;
- `1a_refactor_atm_leadtime_acc_skill_map.ipynb`: atmospheric prepared anomaly bundles and prepared observations;
- `1b_refactor_lnd_leadtime_acc_skill_map.ipynb`: prepared land forecast and reference bundles.

Those notebooks are not execution prerequisites: `inputs.mode="auto"` prepares only the selected products that are unavailable.

**Input policy:** by default this notebook reuses compatible analysis-ready products and prepares only missing or incompatible selected inputs. Set `CONFIG["inputs"]["mode"]` to `"require"` for an archive-free cache-only run or `"rebuild"` to regenerate the selected dependencies.

For each system, initialization month, and lead, the notebook estimates:

1. the observed teleconnection map: correlation of the observed SST index with the observed downstream anomaly;
2. the forecast teleconnection map: correlation of the ensemble-mean forecast SST index with the ensemble-mean downstream anomaly across initialization years;
3. scalar fidelity metrics comparing forecast and observed maps: spatial pattern correlation, centered spatial RMSE, regression slope/amplitude ratio, sign agreement, significant-area fractions, and sample counts.

Correlation maps, p-values, and scalar metrics are saved to a provenance-rich NetCDF file. A lead-summary figure and selected map panels are also saved.

In [ ]:
import os
import sys
from pathlib import Path

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = (
    [Path(_repo_override).expanduser().resolve()]
    if _repo_override
    else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
)
REPO_ROOT = next((p for p in _repo_candidates if (p / "workflows" / "diagnostics").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
import importlib
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True
except ImportError:
    HAVE_CARTOPY = False

try:
    from scipy import stats as scipy_stats
except ImportError as exc:
    raise ImportError("This notebook requires scipy for p-values") from exc

from esp_lab.leadtime_plot_utils import seasonal_label
from esp_lab.utils import colormap_utils as mycolors
from workflows.diagnostics import sst_teleconnections as telecon
telecon = importlib.reload(telecon)  # pick up local workflow edits in a live kernel

from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

%matplotlib inline


## Managed Dask resources

A local production run uses worker processes with the shared-node safety cap.
Rerunning this cell closes any previous notebook cluster and tracked datasets first.


In [ ]:
import dask
from dask.distributed import (
    wait,
    get_client,
)
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

dask.__version__

# User-adjustable Dask settings (shared login nodes are capped at four workers).
DASK_SETTINGS = {
    "enabled": True,
    "cluster_type": "local",
    "workers": 12,
    "memory_limit": "4GB",
}

dask_cfg = DaskConfig(
    cluster_type=DASK_SETTINGS["cluster_type"],
    workers=DASK_SETTINGS["workers"],
    memory_limit=DASK_SETTINGS["memory_limit"],
)
cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(dask_cfg) if DASK_SETTINGS["enabled"] else (None, None),
)
if client is not None:
    display(client)


## 1. User configuration

Normally only the explicit `sst_index`, `tel_var`, `inputs`, and `regrid` choices need editing. Each run analyzes one intentional SST-index/downstream-variable teleconnection pair. The `regrid` block follows the shared preparation contract: 3b reuses a compatible cache or, in `inputs.mode="auto"`, prepares the selected raw input on that grid. The seasonal contract uses centered three-month means.

In [ ]:
# Explicit teleconnection pair selected for this run.
sst_index = "Nino3.4"
tel_var = "TS" #"PRECT" #"TREFHT"
CONFIG = {
    "paths": {
        "diag_root": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
        "output_dir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/teleconnections",
        "figure_dir": "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/teleconnections",
    },
    "inputs": {
        "mode": "auto",  # auto | require | rebuild
        "initialization_years": (1980, 2011),
        "raw_model_root": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
        "observation_root": "/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series",
        "ensemble_member_count": 10,
        "monthly_nlead": 24,
        "workers": 4,
        "sst_land_mask": True,
    },
    "selection": {
        "upstream_index": sst_index,
        "downstream_variable": tel_var,
        "systems": ["E3SM-FOSIRL", "E3SM-Reanalysis", "E3SM-4DEnVarOcn"],
        "init_months": [5, 11],
        "leads": "all",  # all common complete seasonal leads, or a list of stored L values
        "verification_years": (1981, 2011),
        "climatology_years": (1981, 2010),
    },
    # Desired grid contract for either compatible-cache reuse or on-demand preparation.
    "regrid": {
        "target_dlat": 5.0,
        "target_dlon": 5.0,
        "method": "conservative",
        "periodic": True,
    },
    "analysis": {
        "detrend": True,
        "alpha": 0.10, # significance level
        "minimum_years": 20,
        "latitude_bounds": (-80.0, 80.0),
        "area_weighted": True,
        "sign_agreement_threshold": 0.0,
    },
    "cache": {
        "mode": "auto",  # auto | rebuild | require
        "allow_ambiguous_matches": True,
    },
    "figures": {
        "map_leads": None,  # Stored L coordinates; None = every available seasonal lead
        "dpi": 300,
        "cmap": "blue2red_acc",
        "correlation_limits": (-1.0, 1.0),
        "correlation_interval": 0.1,
        "correlation_cutoff": 0.5,
        "colorbar_ticks": (-0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9),
        "show_significant_only": True,
    },
}

E3SM_CASES = telecon.E3SM_CASES
SST_INDEX_REGIONS = telecon.SST_INDEX_REGIONS
DOWNSTREAM_VARIABLES = telecon.DOWNSTREAM_VARIABLES

index_name = CONFIG["selection"]["upstream_index"]
if index_name not in SST_INDEX_REGIONS:
    raise ValueError(f"Unknown SST index {index_name!r}; choose from {list(SST_INDEX_REGIONS)}")
if tel_var not in DOWNSTREAM_VARIABLES:
    raise ValueError(f"Unconfigured downstream variable {tel_var!r}; choose from {list(DOWNSTREAM_VARIABLES)}")
if CONFIG["inputs"]["mode"] not in {"auto", "rebuild", "require"}:
    raise ValueError("inputs.mode must be auto, rebuild, or require")
if CONFIG["cache"]["mode"] not in {"auto", "rebuild", "require"}:
    raise ValueError("cache.mode must be auto, rebuild, or require")


## 2. Cache registry and discovery

This registry separates scientific names from physical files. Atmospheric prepared bundles contain `anomaly`, `climatology`, and `time`; land bundles contain the configured field plus `time`. Observation caches contain `observation` (atmosphere) or the land reference variable.

Hashed/versioned products are selected by metadata, not merely by filename. If more than one compatible candidate remains, discovery stops unless the user explicitly permits choosing the newest match.

In [ ]:
DIAG_ROOT = Path(CONFIG["paths"]["diag_root"])
OUTPUT_DIR = Path(CONFIG["paths"]["output_dir"])
FIGURE_DIR = Path(CONFIG["paths"]["figure_dir"])

# Expose reusable workflow helpers
select_cache = telecon.select_cache
upstream_paths = telecon.upstream_sst_paths
resolve_downstream_paths = telecon.resolve_downstream_paths


In [ ]:
# Reuse or prepare selected upstream products, then validate the inventory.
inventory = telecon.ensure_upstream_products(CONFIG)
display(inventory)

missing = inventory.query("status == 'missing'")
if not missing.empty:
    raise FileNotFoundError(
        "Required upstream products remain missing after applying inputs.mode.\n"
        + missing.to_string(index=False)
    )
skipped = inventory.query("status == 'skipped'")
if not skipped.empty:
    print(f"Note: {len(skipped)} combination(s) skipped (e.g. system has no land component).")


## 3. Alignment and anomaly helpers

Alignment is by **target year derived from each cache's valid-time coordinate**, never by positional index. This matters because initialization year and verification year differ at long leads. Forecast SST and downstream fields must share the same initialization-year/lead grid and valid target season.

The 1a `anomaly` field is already lead-dependent drift corrected. Land caches are normalized here only when they are not explicitly marked as anomalies. Observed fields and indices are converted to monthly-climatology anomalies over the configured climatology window, then sampled at the forecast target dates.

In [ ]:
time_year_month = telecon.time_year_month
parse_init_years = telecon.parse_init_years
lead_signature = telecon.lead_signature
match_leads = telecon.match_leads
linear_detrend = telecon.linear_detrend
monthly_anomaly = telecon.monthly_anomaly
lead_anomaly = telecon.lead_anomaly
observed_at_valid_time = telecon.observed_at_valid_time
corr_and_p = telecon.corr_and_p
weighted_spatial_metrics = telecon.weighted_spatial_metrics


## 4. Teleconnection computation

The scalar comparison domain is the intersection of finite forecast/observed maps and the configured latitude band. Cosine-latitude weights are used by default. The forecast map is based on the ensemble mean for both the SST index and downstream field; this measures the predictable, forced teleconnection rather than within-ensemble weather covariance.

A future extension can add member-wise distributions or paired bootstrap uncertainty without changing the saved core dimensions.

In [ ]:
def compute_one(system, init_month, variable):
    """Compute lead-dependent teleconnection metrics for one (system, init_month, variable) case."""
    return telecon.compute_system_teleconnection(system, init_month, variable, CONFIG)


## 5. Cache metrics

The output key includes all scientific choices. `auto` reuses an exact compatible metrics file; `rebuild` replaces it; `require` refuses to compute if it is absent. Source paths, modification times, and sizes enter the fingerprint so upstream cache changes invalidate downstream teleconnection metrics.

In [ ]:
metrics_ds, out_file, cache_status = telecon.ensure_teleconnection_dataset(CONFIG, inventory=inventory)
fingerprint = metrics_ds.attrs.get("fingerprint", "latest")
print(f"Teleconnection metrics ({cache_status}): {out_file}")
display(metrics_ds)


## 6. Analysis figures

- The `map panels` are generated first, placing the observed reference beside every forecast system for each initialization month. The fidelity summary follows, with each initialization month in a separate row. When `show_significant_only` is enabled, grid cells that do not meet the configured pointwise significance level are masked; interpret this as descriptive unless a field-significance/FDR extension is added.
- `pattern_correlation` asks whether the forecast reproduces the geographical shape of the observed teleconnection.
- `centered_rmse` measures spatial pattern error after removing each map's area-weighted mean.
- `amplitude_ratio` and `regression_slope` distinguish weak/strong patterns and sign reversal.
- `sign_agreement_fraction` is an intuitive spatial consistency measure.
- Pointwise p-values are saved, but multiple-testing control and field significance are not claimed.

In [ ]:
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.offsetbox import AnchoredText

def initialization_label(month):
    return pd.Timestamp(2000, int(month), 1).strftime("%b").upper()


def display_lead(lead):
    """Convert the stored centered-season L coordinate to 1a's displayed lead."""
    return int(lead) - 2

def draw_map(ax, da, *, pvalue=None):
    lat = "lat" if "lat" in da.coords else "latitude"
    lon = "lon" if "lon" in da.coords else "longitude"
    lower, upper = correlation_limits
    interval = correlation_interval
    levels = np.arange(lower, upper + 0.5 * interval, interval)
    cmap = (
        mycolors.blue2red_acc_cmap(levels, correlation_cutoff)
        if cmap_name == "blue2red_acc" else plt.get_cmap(cmap_name)
    )
    norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)
    kwargs = dict(
        x=lon, y=lat, ax=ax, cmap=cmap, norm=norm,
        add_colorbar=False, transform=ccrs.PlateCarree() if HAVE_CARTOPY else None
    )
    image = da.plot(**{k: v for k, v in kwargs.items() if v is not None})

    # Overlay significance test as dot hatch instead of masking
    if show_significant and pvalue is not None:
        sig = (pvalue.values <= alpha_sig).astype(float)
        if np.any(sig > 0):
            ax.contourf(
                da[lon].values, da[lat].values, sig,
                levels=[0.5, 1.5],
                colors="none",
                hatches=[sig_hatch_pattern],
                transform=ccrs.PlateCarree() if HAVE_CARTOPY else None,
            )

    if HAVE_CARTOPY:
        ax.set_global()
        ax.coastlines(linewidth=coastline_linewidth, color=coastline_color)
        ax.add_feature(cfeature.BORDERS, linewidth=border_linewidth, edgecolor=border_edgecolor)
        ax.gridlines(linewidth=gridline_linewidth, color=gridline_color, alpha=gridline_alpha, linestyle=gridline_linestyle)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")  # suppress xarray's scalar-coordinate title
    return image


In [ ]:
# =========================================================================
# 6. Multi-Panel Teleconnection Correlation Maps - Setup Parameters
# =========================================================================
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Base font size and proportional scaling factors for robust typography
fontz = 18
scale_suptitle = 1.20   # Figure supertitle
scale_header = 1.08     # Month-group headers
scale_col_title = 0.88  # Case/reference column titles
scale_label = 0.92      # Colorbar label
scale_tick = 0.72       # Longitude/latitude & colorbar ticks
scale_badge = 0.72      # Lead badge & missing placeholder text

fs_suptitle = fontz * scale_suptitle
fs_header = fontz * scale_header
fs_col_title = fontz * scale_col_title
fs_label = fontz * scale_label
fs_tick = fontz * scale_tick
fs_badge = fontz * scale_badge
gap_ratio = 0.22
lon_ticks = [-120, 0, 120]
lat_ticks = [-60, -30, 0, 30, 60]

configured_leads = CONFIG["figures"].get("map_leads", None)
available_leads = [int(lead) for lead in metrics_ds.L.values]
map_leads = (
    available_leads if configured_leads is None
    else [int(lead) for lead in configured_leads if int(lead) in available_leads]
)
plot_systems = [
    system for system in CONFIG["selection"]["systems"]
    if system in set(metrics_ds.system.values.astype(str))
]
plot_months = [
    int(month) for month in CONFIG["selection"]["init_months"]
    if int(month) in set(map(int, metrics_ds.init_month.values))
]
if not plot_months:
    raise ValueError("No configured initialization months are available")

correlation_limits = CONFIG["figures"].get("correlation_limits", (-1.0, 1.0))
correlation_interval = CONFIG["figures"].get("correlation_interval", 0.1)
correlation_cutoff = CONFIG["figures"].get("correlation_cutoff", 0.5)
colorbar_ticks = CONFIG["figures"].get("colorbar_ticks", (-0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9))
cmap_name = CONFIG["figures"].get("cmap", "blue2red_acc")
figure_dpi = CONFIG["figures"].get("dpi", 300)
show_significant = CONFIG["figures"].get("show_significant_only", True)
alpha_sig = CONFIG["analysis"].get("alpha", 0.10)

# Layout and sizing (user-adjustable setup parameters)
# Set fig_size to an explicit (width, height) tuple, or None to auto-compute from panel dimensions
fig_size = None  # e.g. (22.0, 9.5) or None for dynamic calculation
panel_width = 3.3
panel_height = 1.75
fig_base_width = 0.8
fig_base_height = 2.2
gridspec_left = 0.035
gridspec_right = 0.985
gridspec_bottom = 0.08
gridspec_top = 0.90
gridspec_hspace = 0.07
gridspec_wspace = 0.03

# Cartopy map styling
coastline_linewidth = 0.5
coastline_color = "0.25"
border_linewidth = 0.2
border_edgecolor = "0.45"
gridline_linewidth = 0.25
gridline_color = "0.65"
gridline_alpha = 0.4
gridline_linestyle = ":"
sig_hatch_pattern = "..."

index_display = index_name.replace("Nino", "Niño")

for variable in metrics_ds.variable.values:
    variable_leads = [
        lead for lead in map_leads
        if not metrics_ds.model_correlation.sel(variable=variable, L=lead).isnull().all()
    ]
    if not variable_leads:
        print(f"Skipped {variable}: no model correlation maps are available")
        continue
    nrows = len(variable_leads)
    nsys = len(plot_systems)
    group_width = nsys + 1  # observed reference plus forecast systems
    width_ratios = []
    for month_index in range(len(plot_months)):
        width_ratios.extend([1.0] * group_width)
        if month_index < len(plot_months) - 1:
            width_ratios.append(gap_ratio)
    total_cols = len(width_ratios)
    projection = ccrs.PlateCarree() if HAVE_CARTOPY else None
    current_figsize = (
        fig_size
        if fig_size is not None
        else (panel_width * len(plot_months) * group_width + fig_base_width, panel_height * nrows + fig_base_height)
    )
    fig = plt.figure(figsize=current_figsize)
    grid = fig.add_gridspec(
        nrows, total_cols, width_ratios=width_ratios,
        left=gridspec_left, right=gridspec_right, bottom=gridspec_bottom, top=gridspec_top,
        hspace=gridspec_hspace, wspace=gridspec_wspace,
    )
    axes = {}
    image = None
    for month_index, init_month in enumerate(plot_months):
        column_offset = month_index * (group_width + 1)
        for row, lead in enumerate(variable_leads):
            ax = fig.add_subplot(grid[row, column_offset], projection=projection)
            axes[(init_month, "Reference", lead)] = ax
            reference = reference_pvalue = None
            for system in plot_systems:
                candidate = metrics_ds.sel(
                    variable=variable, system=system, init_month=init_month, L=lead
                )
                if not candidate.observed_correlation.isnull().all():
                    reference = candidate.observed_correlation
                    reference_pvalue = candidate.observed_pvalue
                    break
            if reference is None:
                if HAVE_CARTOPY:
                    ax.set_global()
                ax.set_facecolor("0.95")
                ax.text(
                    0.5, 0.5, "Ref N/A",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=fs_badge, color="0.45",
                )
            else:
                image = draw_map(ax, reference, pvalue=reference_pvalue)
            for sys_index, system in enumerate(plot_systems):
                col = column_offset + 1 + sys_index
                ax = fig.add_subplot(grid[row, col], projection=projection)
                axes[(init_month, system, lead)] = ax
                sub = metrics_ds.sel(
                    variable=variable, system=system, init_month=init_month, L=lead
                )
                if sub.model_correlation.isnull().all():
                    if HAVE_CARTOPY:
                        ax.set_global()
                    ax.set_facecolor("0.95")
                    ax.text(
                        0.5, 0.5, "N/A",
                        ha="center", va="center", transform=ax.transAxes,
                        fontsize=fs_badge, color="0.45",
                    )
                else:
                    image = draw_map(ax, sub.model_correlation, pvalue=sub.model_pvalue)

    # Label rows with lead time on the leftmost column of each month group
    for month_index, init_month in enumerate(plot_months):
        for row, lead in enumerate(variable_leads):
            disp = display_lead(lead)
            label = (
                f"lead {disp}: {seasonal_label(init_month, disp)}"
                if HAVE_CARTOPY else f"lead {disp}"
            )
            ax = axes[(init_month, "Reference", lead)]
            badge = AnchoredText(
                label, loc="lower left",
                prop=dict(size=fs_badge, weight="bold", color="black"),
                frameon=True, pad=0.15, borderpad=0.25,
            )
            badge.patch.set(facecolor="white", edgecolor="none", alpha=0.85)
            ax.add_artist(badge)

    # Column titles along the top row
    top_lead = variable_leads[0]
    for month_index, init_month in enumerate(plot_months):
        ref_ax = axes[(init_month, "Reference", top_lead)]
        ref_name = DOWNSTREAM_VARIABLES.get(variable, {}).get("reference", "Observed")
        ref_ax.set_title(f"Observed ({ref_name})", fontsize=fs_col_title, fontweight="bold", pad=8)
        for system in plot_systems:
            sys_ax = axes[(init_month, system, top_lead)]
            title = E3SM_CASES[system].get("display_name", system)
            sys_ax.set_title(title, fontsize=fs_col_title, fontweight="bold", pad=8)

    # Month-group supertitles
    for month_index, init_month in enumerate(plot_months):
        left_ax = axes[(init_month, "Reference", top_lead)]
        right_ax = axes[(init_month, plot_systems[-1], top_lead)]
        fig.canvas.draw_idle()
        x0 = left_ax.get_position().x0
        x1 = right_ax.get_position().x1
        fig.text(
            0.5 * (x0 + x1), 0.94,
            f"{initialization_label(init_month)} initialization",
            ha="center", va="bottom",
            fontsize=fs_header, fontweight="bold",
        )

    # Ticks along left/bottom outer boundaries
    for (init_month, col_key, lead), ax in axes.items():
        row = variable_leads.index(lead)
        is_bottom = (row == nrows - 1)
        is_left = (col_key == "Reference")
        if HAVE_CARTOPY:
            ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
            ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())
            ax.xaxis.set_major_formatter(
                LongitudeFormatter(zero_direction_label=True)
                if is_bottom else plt.NullFormatter()
            )
            ax.yaxis.set_major_formatter(
                LatitudeFormatter() if is_left else plt.NullFormatter()
            )
            ax.tick_params(
                labelsize=fs_tick,
                labelbottom=is_bottom,
                labelleft=is_left,
                length=3,
            )

    # Shared colorbar
    if image is not None:
        cbar_ax = fig.add_axes([0.22, 0.025, 0.56, 0.022])
        cbar = fig.colorbar(image, cax=cbar_ax, orientation="horizontal", ticks=colorbar_ticks)
        cbar.ax.tick_params(labelsize=fs_tick)
        cbar.set_label(
            f"Correlation: {index_display} SST index ↔ {variable} anomaly",
            fontsize=fs_label,
            labelpad=6,
        )

    fig.suptitle(
        f"{index_display}–{variable} Teleconnection Maps",
        fontsize=fs_suptitle,
        fontweight="bold",
        y=0.985,
    )

    path = FIGURE_DIR / (
        f"teleconnection_maps_{index_name.replace('.', '')}_{variable}.png"
    )
    fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

In [ ]:
# =========================================================================
# 6. Teleconnection Fidelity Summary - Setup Parameters
# =========================================================================
# Base font size and proportional scaling factors for robust typography
fontz = 14
scale_suptitle = 1.0   # Figure supertitle
scale_title = 0.95      # Subplot column titles
scale_label = 0.9      # Axis labels
scale_legend = 0.85     # Legend text
scale_tick = 0.8       # Tick labels

fs_suptitle = fontz * scale_suptitle
fs_title = fontz * scale_title
fs_label = fontz * scale_label
fs_legend = fontz * scale_legend
fs_tick = fontz * scale_tick
summary_metrics = ["pattern_correlation", "centered_rmse", "amplitude_ratio", "sign_agreement_fraction"]
ylabels = ["Pattern correlation", "Centered RMSE", "Amplitude ratio", "Sign agreement"]
plot_systems = [
    system for system in CONFIG["selection"]["systems"]
    if system in set(metrics_ds.system.values.astype(str))
]
plot_months = [
    int(month) for month in CONFIG["selection"]["init_months"]
    if int(month) in set(map(int, metrics_ds.init_month.values))
]
figure_dpi = CONFIG["figures"].get("dpi", 300)

METHOD_STYLES = {
    "E3SM-4DEnVarOcn": {"short_name": "4DEnVarOcn", "color": "tab:purple", "marker": "o"},
    "E3SM-FOSIRL": {"short_name": "FOSIRL", "color": "black", "marker": "s"},
    "E3SM-Reanalysis": {"short_name": "Reanalysis", "color": "tab:blue", "marker": "D"},
}

# Figure layout and visual styling (user-adjustable setup parameters)
fig_size = (8.5, 10.5)  # explicit (width, height) figure size
marker_size = 6
line_width = 1.5
grid_color = "0.88"
grid_linewidth = 0.6
grid_alpha = 0.7
grid_linestyle = "-"

# Reference lines styling
ref_amplitude_ratio = 1.0
ref_amplitude_color = "0.55"
ref_amplitude_linestyle = "--"
ref_amplitude_linewidth = 0.9

ref_pattern_corr = 0.0
ref_pattern_color = "0.65"
ref_pattern_linestyle = ":"
ref_pattern_linewidth = 0.8

ref_sign_agreement = 0.5
ref_sign_color = "0.65"
ref_sign_linestyle = ":"
ref_sign_linewidth = 0.8

# Explicit y-axis range controls per metric row (None for auto-scale, or (ymin, ymax))
ylim_pattern_correlation = None #(0.2, 1.0)
ylim_centered_rmse = None #(0, 0.5)  # None = auto-scale based on data range
ylim_amplitude_ratio = None #(0.6, 1.8)
ylim_sign_agreement = None #(0.4, 1.0)

metric_ylimits = {
    "pattern_correlation": ylim_pattern_correlation,
    "centered_rmse": ylim_centered_rmse,
    "amplitude_ratio": ylim_amplitude_ratio,
    "sign_agreement_fraction": ylim_sign_agreement,
}

# Layout padding, legend, and bottom x-axis tick labels
tight_layout_rect = (0.02, 0.055, 0.98, 0.96)
legend_bbox = (0.5, 0.015)
xlabel_text = "Lead time (months)"  # bottom x-axis title (or None / "" to hide)
show_seasonal_lead_ticks = True     # True for "{lead}:{season}" format (e.g. 1:DJF, 4:MAM)
xtick_rotation = 0                  # 0 for horizontal (e.g. "1:DJF"), or 30/45 for angled

index_display = index_name.replace("Nino", "Niño")

for variable in metrics_ds.variable.values:
    valid_leads = [
        int(lead) - 2 for lead in metrics_ds.L.values
        if np.any(np.isfinite(metrics_ds.pattern_correlation.sel(variable=variable, L=lead)))
    ]

    nrows = len(summary_metrics)
    ncols = len(plot_months)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=fig_size,
        sharex="col",
        sharey="row",
        squeeze=False,
    )

    for col, init_month in enumerate(plot_months):
        month_label = pd.Timestamp(2000, int(init_month), 1).strftime("%b")
        col_title = f"{month_label} Initialization"

        for row, (metric, ylabel) in enumerate(zip(summary_metrics, ylabels)):
            ax = axes[row, col]

            # Reference lines for physical context
            if metric == "amplitude_ratio":
                ax.axhline(
                    ref_amplitude_ratio,
                    color=ref_amplitude_color,
                    linestyle=ref_amplitude_linestyle,
                    linewidth=ref_amplitude_linewidth,
                    alpha=0.7,
                    zorder=1
                )
            elif metric == "pattern_correlation":
                ax.axhline(
                    ref_pattern_corr,
                    color=ref_pattern_color,
                    linestyle=ref_pattern_linestyle,
                    linewidth=ref_pattern_linewidth,
                    alpha=0.6,
                    zorder=1
                )
            elif metric == "sign_agreement_fraction":
                ax.axhline(
                    ref_sign_agreement,
                    color=ref_sign_color,
                    linestyle=ref_sign_linestyle,
                    linewidth=ref_sign_linewidth,
                    alpha=0.6,
                    zorder=1
                )

            for system in metrics_ds.system.values:
                sys_str = str(system)
                if sys_str not in E3SM_CASES:
                    continue
                series = metrics_ds[metric].sel(
                    variable=variable, system=sys_str, init_month=init_month
                )
                if series.isnull().all():
                    continue

                style = METHOD_STYLES.get(sys_str, {
                    "color": E3SM_CASES[sys_str]["color"],
                    "marker": "o",
                    "short_name": E3SM_CASES[sys_str]["display_name"],
                })

                leads_display = series.L.astype(int) - 2
                ax.plot(
                    leads_display, series,
                    marker=style["marker"],
                    markersize=marker_size,
                    linewidth=line_width,
                    color=style["color"],
                    label=E3SM_CASES[sys_str]["display_name"],
                    zorder=3,
                )

            ax.grid(True, linestyle=grid_linestyle, color=grid_color, linewidth=grid_linewidth, alpha=grid_alpha)
            ax.tick_params(labelsize=fs_tick)

            # Explicit y-axis range if configured
            ylim = metric_ylimits.get(metric)
            if ylim is not None:
                ax.set_ylim(ylim)

            # Column headers on top row
            if row == 0:
                ax.set_title(col_title, fontsize=fs_title, fontweight="bold", pad=10)

            # Row y-labels on left column
            if col == 0:
                ax.set_ylabel(ylabel, fontsize=fs_label, fontweight="bold")

            # X-axis label and ticks on bottom row
            if row == nrows - 1:
                if xlabel_text:
                    ax.set_xlabel(xlabel_text, fontsize=fs_label)
                if valid_leads:
                    if show_seasonal_lead_ticks:
                        xticklabs = [f"{l}:{seasonal_label(init_month, l)}" for l in valid_leads]
                    else:
                        xticklabs = [str(l) for l in valid_leads]
                    ax.set_xticks(valid_leads)
                    ax.set_xticklabels(xticklabs, fontsize=fs_tick, rotation=xtick_rotation)
                    ax.set_xlim(valid_leads[0] - 0.8, valid_leads[-1] + 0.8)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(
            handles, labels,
            loc="lower center",
            ncol=len(handles),
            fontsize=fs_legend,
            frameon=True,
            framealpha=0.9,
            edgecolor="0.8",
            bbox_to_anchor=legend_bbox,
        )

    fig.suptitle(
        f"{index_display}–{variable} Teleconnection Fidelity Summary",
        fontsize=fs_suptitle,
        fontweight="bold",
        y=0.985,
    )
    plt.tight_layout(rect=tight_layout_rect)
    path = FIGURE_DIR / f"teleconnection_{index_name.replace('.', '')}_{variable}_summary.png"
    fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

## 7. Interpretation and recommended extensions

### Taylor diagram synthesis
The Taylor diagram below synthesizes three complementary pattern fidelity metrics simultaneously:
- **Azimuthal angle ($\theta = \arccos(r)$)**: Pattern correlation with the observed teleconnection.
- **Radial distance ($R = \sigma_m / \sigma_o$)**: Spatial amplitude ratio (relative spatial variability).
- **Distance from Reference star ($(R=1.0, \theta=0)$)**: Centered RMS Error (CRMSE).

A perfect forecast teleconnection pattern would coincide with the gold **Reference (Obs)** star at $(1.0, 0^\circ)$.

- `pattern_correlation` asks whether the forecast reproduces the geographical shape of the observed teleconnection.
- `centered_rmse` measures spatial pattern error after removing each map's area-weighted mean.
- `amplitude_ratio` and `regression_slope` distinguish weak/strong patterns and sign reversal.
- `sign_agreement_fraction` is an intuitive spatial consistency measure.
- Pointwise p-values are saved, but multiple-testing control and field significance are not claimed.

Recommended phase-2 additions are paired bootstrap confidence intervals over years, member-wise teleconnection distributions, partial correlations controlling for another SST index, lagged index/response seasons, and FDR or field-significance testing. Those should be new configuration options rather than changes to the cache-first input contract.

In [ ]:
# =========================================================================
# 7. Taylor Diagram Synthesis - Setup Parameters
# =========================================================================
# Base font size and proportional scaling factors for robust typography
fontz = 14
scale_suptitle = 1.0    # Figure supertitle
scale_title = 0.95      # Subplot title
scale_label = 0.90       # Radial axis label
scale_legend = 0.9      # Legend text
scale_tick = 0.90        # Correlation arc & radial ticks, arc title
scale_annotation = 0.60  # Lead trajectory annotations (L1, L19)
scale_contour = 0.5      # CRMSE contour labels

fs_suptitle = fontz * scale_suptitle
fs_title = fontz * scale_title
fs_label = fontz * scale_label
fs_legend = fontz * scale_legend
fs_tick = fontz * scale_tick
fs_annotation = fontz * scale_annotation
fs_contour = fontz * scale_contour

x_label = r"Normalized Standard Deviation ($\sigma_m / \sigma_o$)"

r_max = 2.0
r_ticks = [0.5, 1.0, 1.5]
crmse_levels = [0.25, 0.50, 0.75, 1.00, 1.25]
marker_size_range = (4.5, 8.5)
marker_alpha_range = (0.45, 0.95)
r_color="0.6"
r_fontweight="semibold"
r_label="Pattern Correlation"
r_ha="center"
r_va="bottom"
            
connect_leads = True
figure_dpi = CONFIG["figures"].get("dpi", 300)

# Angular extent and correlation tick marks (user-adjustable setup parameters)
thetamax_mode = "auto"  # "auto" (90 deg if all corr >= 0 else 180 deg) or explicit 90 / 180
corr_ticks_90 = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99])
corr_ticks_180 = np.array([-0.9, -0.7, -0.5, -0.3, -0.1, 0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99])
th_ref_90 = np.linspace(0, np.pi / 2, 100)
th_ref_180 = np.linspace(0, np.pi, 100)
corr_label_angle_90 = np.deg2rad(30)
corr_label_rotation_90 = -55
corr_label_angle_180 = np.deg2rad(60)
corr_label_rotation_180 = 0

# Visual styling & figure layout (user-adjustable setup parameters)
# Set fig_size to an explicit (width, height) tuple, or None to auto-compute per month
fig_size = (12.8, 6.4)  # e.g. (12.8, 6.4) or None for dynamic calculation
fig_width_per_month = 6.4
fig_height = 6.2
grid_color = "0.88"
grid_linewidth = 0.6
grid_linestyle = "-"

# Reference observation point & circle
ref_marker = "*"
ref_marker_size = 12
ref_color = "gold"
ref_edgecolor = "black"
ref_edgewidth = 0.9
ref_circle_color = "0.35"
ref_circle_linestyle = "--"
ref_circle_linewidth = 0.9
ref_circle_alpha = 0.6

# CRMSE contour styling
crmse_colors = "0.72"
crmse_linestyles = ":"
crmse_linewidths = 0.7
crmse_alpha = 0.7
crmse_grid_points = 150

# Model trajectory styling & collision-free lead annotations
lead_line_linewidth = 1.1
lead_line_alpha = 0.35
show_lead_annotations = True  # True to annotate endpoints (L1, last lead), False for clean trajectory
lead_annotation_box = dict(
    boxstyle="round,pad=0.10",
    facecolor="white",
    edgecolor="0.80",
    linewidth=0.4,
    alpha=0.88,
)

# Subplot margins, titles, and legend layout
title_pad = 14
xlabel_pad = 14
suptitle_y = 0.98
subplots_adjust_bottom = 0.15
subplots_adjust_top = 0.82
subplots_adjust_wspace = 0.25
subplots_adjust_left = 0.06
subplots_adjust_right = 0.94
legend_bbox = (0.5, 0.015)

plot_months = [
    int(month) for month in CONFIG["selection"]["init_months"]
    if int(month) in set(map(int, metrics_ds.init_month.values))
]

METHOD_STYLES = {
    "E3SM-FOSIRL": {
        "short_name": "FOSIRL",
        "color": "black",
        "marker": "s",
        "offset_start": (0, 8),
        "offset_end": (0, 8),
    },
    "E3SM-Reanalysis": {
        "short_name": "Reanalysis",
        "color": "tab:blue",
        "marker": "D",
        "offset_start": (9, -8),
        "offset_end": (9, 6),
    },
    "E3SM-4DEnVarOcn": {
        "short_name": "4DEnVarOcn",
        "color": "tab:purple",
        "marker": "o",
        "offset_start": (-9, 7),
        "offset_end": (-9, -8),
    },
}

index_display = index_name.replace("Nino", "Niño")

for variable in metrics_ds.variable.values:
    if thetamax_mode == "auto":
        min_corr = float(
            metrics_ds.pattern_correlation.sel(variable=variable).min(skipna=True)
        )
        thetamax = 180 if min_corr < 0 else 90
    else:
        thetamax = int(thetamax_mode)

    if thetamax == 90:
        corr_ticks = corr_ticks_90
        th_ref = th_ref_90
        corr_label_angle = corr_label_angle_90
        corr_label_rotation = corr_label_rotation_90
    else:
        corr_ticks = corr_ticks_180
        th_ref = th_ref_180
        corr_label_angle = corr_label_angle_180
        corr_label_rotation = corr_label_rotation_180

    current_figsize = (
        fig_size
        if fig_size is not None
        else (fig_width_per_month * len(plot_months), fig_height)
    )
    fig = plt.figure(figsize=current_figsize)
    legend_handles = []
    legend_labels = []

    for idx, init_month in enumerate(plot_months):
        ax = fig.add_subplot(1, len(plot_months), idx + 1, projection="polar")
        ax.set_thetamin(0)
        ax.set_thetamax(thetamax)
        ax.set_ylim(0, r_max)

        # Lighten the background grid substantially
        ax.grid(True, color=grid_color, linewidth=grid_linewidth, linestyle=grid_linestyle)

        # Correlation ticks on outer arc
        theta_ticks = np.arccos(corr_ticks)
        ax.set_xticks(theta_ticks)
        ax.set_xticklabels([f"{c:.2g}" for c in corr_ticks], fontsize=fs_tick, color="0.25")

        # Radial ticks
        ax.set_yticks(r_ticks)
        ax.set_yticklabels([f"{t:.1f}" for t in r_ticks], fontsize=fs_tick, color="0.25")

        # Radial axis label
        ax.set_xlabel(
            x_label,
            fontsize=fs_label,
            labelpad=xlabel_pad,
            color="0.2",
        )

        # Subtle correlation label along the arc
        ax.text(
            corr_label_angle,
            r_max,
            r_label,
            fontsize=fs_tick,
            ha=r_ha,
            va=r_va,
            rotation=corr_label_rotation,
            color=r_color,
            fontweight=r_fontweight,
        )

        # Contours of constant centered RMS error (CRMSE)
        rs = np.linspace(0, r_max, crmse_grid_points)
        thetas = np.linspace(0, np.deg2rad(thetamax), crmse_grid_points)
        R_grid, TH_grid = np.meshgrid(rs, thetas)
        X_grid = R_grid * np.cos(TH_grid)
        Y_grid = R_grid * np.sin(TH_grid)
        E_grid = np.sqrt((X_grid - 1.0) ** 2 + Y_grid ** 2)
        cs = ax.contour(
            TH_grid,
            R_grid,
            E_grid,
            levels=crmse_levels,
            colors=crmse_colors,
            linestyles=crmse_linestyles,
            linewidths=crmse_linewidths,
            alpha=crmse_alpha,
        )
        ax.clabel(cs, inline=True, fontsize=fs_contour, fmt="%.2f", colors="0.45")

        # Reference circle at normalized standard deviation = 1.0
        ax.plot(
            th_ref,
            np.ones_like(th_ref),
            color=ref_circle_color,
            linestyle=ref_circle_linestyle,
            linewidth=ref_circle_linewidth,
            alpha=ref_circle_alpha,
        )

        # Reference point at (theta=0, r=1.0) - modest gold star
        ref_line, = ax.plot(
            0,
            1.0,
            marker=ref_marker,
            markersize=ref_marker_size,
            color=ref_color,
            markeredgecolor=ref_edgecolor,
            markeredgewidth=ref_edgewidth,
            linestyle="none",
            zorder=6,
        )
        if idx == 0:
            legend_handles.append(ref_line)
            legend_labels.append("Reference (Obs)")

        # Plot model trajectories
        for system in metrics_ds.system.values:
            sys_str = str(system)
            if sys_str not in METHOD_STYLES:
                continue
            style = METHOD_STYLES[sys_str]
            sub = metrics_ds.sel(variable=variable, system=sys_str, init_month=init_month)
            r_vals = sub.amplitude_ratio.values
            p_vals = sub.pattern_correlation.values
            leads = sub.L.values

            valid = (
                np.isfinite(r_vals)
                & np.isfinite(p_vals)
                & (p_vals >= (-1.0 if thetamax == 180 else 0.0))
                & (p_vals <= 1.0)
            )
            if not np.any(valid):
                continue

            thetas_mod = np.arccos(p_vals[valid])
            rs_mod = r_vals[valid]
            leads_valid = leads[valid]
            n_pts = len(leads_valid)

            color = style["color"]
            marker_shape = style["marker"]
            short_name = style["short_name"]

            # Light connecting line
            if connect_leads:
                ax.plot(
                    thetas_mod,
                    rs_mod,
                    color=color,
                    linestyle="-",
                    linewidth=lead_line_linewidth,
                    alpha=lead_line_alpha,
                    zorder=4,
                )

            # Progressive marker size and alpha across lead time
            s_min, s_max = marker_size_range
            a_min, a_max = marker_alpha_range
            sizes = np.linspace(s_min, s_max, n_pts)
            alphas = np.linspace(a_min, a_max, n_pts)

            for k, (th, r, l, sz, al) in enumerate(zip(thetas_mod, rs_mod, leads_valid, sizes, alphas)):
                lead_label = display_lead(l)
                is_endpoint = (k == 0 or k == n_pts - 1)
                ax.plot(
                    th,
                    r,
                    marker=marker_shape,
                    markersize=sz,
                    color=color,
                    alpha=al,
                    markeredgecolor="black" if is_endpoint else color,
                    markeredgewidth=0.8 if is_endpoint else 0.4,
                    zorder=5,
                )
                # Annotate only first and last lead
                if is_endpoint and show_lead_annotations:
                    offset = style.get("offset_start" if k == 0 else "offset_end", (0, 6))
                    ax.annotate(
                        f"L{lead_label}",
                        xy=(th, r),
                        xytext=offset,
                        textcoords="offset points",
                        fontsize=fs_annotation,
                        ha="center",
                        va="center" if offset[1] != 0 else "bottom",
                        color=color,
                        fontweight="bold",
                        bbox=lead_annotation_box,
                        zorder=7,
                    )

            if idx == 0:
                h_line, = ax.plot(
                    [], [],
                    marker=marker_shape,
                    markersize=7,
                    color=color,
                    linestyle="-",
                    linewidth=lead_line_linewidth,
                    markeredgecolor="black",
                    markeredgewidth=0.7,
                )
                legend_handles.append(h_line)
                legend_labels.append(short_name)

        ax.set_title(
            f"{initialization_label(init_month)} initialization",
            fontsize=fs_title,
            fontweight="bold",
            pad=title_pad,
        )

    fig.suptitle(
        f"{index_display}–{variable} Teleconnection Fidelity",
        fontsize=fs_suptitle,
        fontweight="bold",
        y=suptitle_y,
    )

    fig.legend(
        legend_handles,
        legend_labels,
        loc="lower center",
        ncol=len(legend_handles),
        fontsize=fs_legend,
        frameon=True,
        framealpha=0.9,
        edgecolor="0.8",
        bbox_to_anchor=legend_bbox,
    )

    plt.subplots_adjust(
        bottom=subplots_adjust_bottom,
        top=subplots_adjust_top,
        wspace=subplots_adjust_wspace,
        left=subplots_adjust_left,
        right=subplots_adjust_right,
    )
    path = FIGURE_DIR / (
        f"teleconnection_{index_name.replace('.', '')}_{variable}_"
        f"taylor_diagram.png"
    )
    fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

## 8. Cleanup

Close the notebook cluster and release tracked resources.


In [ ]:
close_notebook_resources(globals())
print("Closed teleconnection-workflow Dask resources.")
